# Clase 16 - Ensamblaje de Modelos

---

## Ensamblaje de Modelos

Para problemas complejos, los clasificadores simples (KNN, Decision Trees, Naive Bayes) pueden ser insuficientes. Una alternativa atractiva antes de recurrir a redes neuronales —que requieren grandes volúmenes de datos— es el **ensamblaje de modelos**.

Los modelos ensamblados son **meta-modelos**: combinaciones de muchos modelos simples (*weak learners*) que, individualmente, tienen baja precisión. La hipótesis central es que su combinación produce un modelo más poderoso con la misma cantidad de datos.

Existen tres paradigmas principales:

| Paradigma | Idea central | Ejemplo |
|-----------|-------------|---------|
| **Bagging** | Modelos en paralelo sobre distintas muestras | Random Forest |
| **Boosting** | Modelos secuenciales que corrigen al anterior | XGBoost, LightGBM |
| **Stacking** | Meta-learner que combina modelos heterogéneos | StackingClassifier |

---

## Credit Card Fraud

El dataset consiste en atributos preprocesados con PCA (V1–V28 más el monto de la transacción). La tarea es predecir si una transacción es fraude o no.

Fuente: [creditcardfraud en Kaggle](https://www.kaggle.com/mlg-ulb/creditcardfraud).

> ❓ **Pregunta:** Si un modelo predice *siempre* "no fraude", ¿qué accuracy obtendría? ¿Lo llamarías un buen modelo?

In [ ]:
import pandas as pd

df = pd.read_pickle("../../recursos/2023-01/21_Ensamblaje/creditcard.pickle")
df = df.drop(columns=["Time"])
df

In [ ]:
import plotly.express as px

conteo = df["Class"].value_counts().reset_index()
conteo.columns = ["Clase", "Cantidad"]
conteo["Clase"] = conteo["Clase"].map({0: "No Fraude", 1: "Fraude"})

px.bar(
    conteo,
    x="Clase",
    y="Cantidad",
    color="Clase",
    text="Cantidad",
    title="Distribución de Clases — Credit Card Fraud",
    template="simple_white",
    color_discrete_map={"No Fraude": "#4C72B0", "Fraude": "#DD4444"},
)

In [ ]:
492 / 280000 * 100

In [ ]:
from sklearn.model_selection import train_test_split

features = df.drop(columns=["Class"])
labels = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, shuffle=True, stratify=labels, test_size=0.3, random_state=10
)

In [ ]:
import plotly.graph_objects as go
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report


def plot_cm(y_true, y_pred, title="Matriz de Confusión"):
    cm = confusion_matrix(y_true, y_pred)
    labels = ["No Fraude", "Fraude"]
    fig = go.Figure(
        go.Heatmap(
            z=cm,
            x=labels,
            y=labels,
            colorscale="Blues",
            text=cm,
            texttemplate="<b>%{text}</b>",
            showscale=False,
        )
    )
    fig.update_layout(
        title=title,
        xaxis_title="Predicción",
        yaxis_title="Real",
        template="simple_white",
        width=450,
        height=380,
    )
    fig.show()
    print(classification_report(y_true, y_pred, target_names=labels))

---

## Bagging (Bootstrap Aggregating)

### La Sabiduría de las Masas

En 1906, el estadístico Francis Galton observó un concurso en una feria: 800 personas apostaban por el peso de un buey. Individualmente, casi nadie acertó. Pero el **promedio de todas las apuestas** estuvo a menos de 500 gramos del peso real.

Este fenómeno, popularizado por James Surowiecki en *The Wisdom of Crowds* (2004), establece que una multitud de estimadores **independientes y diversos** tiende a superar a cualquier experto individual, siempre que sus errores sean no correlacionados y se promedien correctamente.

**Bagging aplica exactamente esta idea al Machine Learning.**

> ❓ **Pregunta:** ¿Qué condiciones deben cumplirse para que "la sabiduría de las masas" funcione? ¿Qué ocurre si todos piensan igual?

### Concepto

**Bagging** entrena $B$ *weak learners* en paralelo, cada uno sobre una muestra distinta del dataset (muestreo con reemplazo, *bootstrap*), y combina sus predicciones.

<br>
<div align='center'>
    <img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/21_Ensamblaje/ensemble_bagging.png?raw=true' width=750/>
</div>
<div align='center'>
    Fuente: <a href='https://en.wikipedia.org/wiki/Bootstrap_aggregating'>Bootstrap aggregating — Wikipedia</a>
</div>
<br>

Para predecir un ejemplo $x$:
- **Regresión:** $\hat{y} = \dfrac{1}{B}\sum_{b=1}^{B}f_b(x)$
- **Clasificación:** clase con más votos entre los $B$ modelos.

> ❓ **Pregunta:** ¿Qué diferencia hay entre entrenar 500 árboles sobre el *mismo* dataset vs. sobre *muestras distintas*?

#### Baseline: DecisionTree

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

pipe_dt = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        ("model", DecisionTreeClassifier(random_state=44)),
    ]
)
pipe_dt.fit(X_train, y_train)
y_pred_dt = pipe_dt.predict(X_test)
plot_cm(y_test, y_pred_dt, "Decision Tree")

#### Random Forest

In [ ]:
pipe_rf = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        ("model", RandomForestClassifier(n_estimators=500, n_jobs=-1, random_state=44)),
    ]
)
pipe_rf.fit(X_train, y_train)
y_pred_rf = pipe_rf.predict(X_test)
plot_cm(y_test, y_pred_rf, "Random Forest (500 árboles)")

#### ¿Cuántos árboles son suficientes?

A medida que se agregan árboles, el modelo mejora pero el beneficio marginal decrece. La siguiente curva muestra cómo evoluciona el F1-Score de la clase fraude al aumentar `n_estimators`.

In [ ]:
from sklearn.metrics import f1_score

n_list = [1, 5, 10, 25, 50, 100, 200, 300, 500]
f1_list = []

for n in n_list:
    p = Pipeline(
        [
            (
                "pre",
                ColumnTransformer(
                    [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
                ),
            ),
            (
                "model",
                RandomForestClassifier(n_estimators=n, n_jobs=-1, random_state=44),
            ),
        ]
    )
    p.fit(X_train, y_train)
    f1_list.append(f1_score(y_test, p.predict(X_test)))

px.line(
    x=n_list,
    y=f1_list,
    markers=True,
    labels={"x": "n_estimators", "y": "F1-Score (Fraude)"},
    title="F1-Score vs n_estimators — Random Forest",
    template="simple_white",
)

#### Out-of-Bag (OOB) Score

Al usar muestreo con reemplazo, cada árbol deja fuera aproximadamente un 37% de los datos. Estos datos *out-of-bag* sirven como validación natural sin necesitar un split adicional.

In [ ]:
rf_oob = RandomForestClassifier(
    n_estimators=500, oob_score=True, n_jobs=-1, random_state=44
)
rf_oob.fit(
    X_train.pipe(
        lambda df: df.assign(Amount=StandardScaler().fit_transform(df[["Amount"]]))
    ),
    y_train,
)
print(f"OOB Score: {rf_oob.oob_score_:.4f}")

#### Importancia de Features

Los modelos de ensamblaje exponen `feature_importances_`: cuánto contribuye cada variable a reducir la impureza en los splits. Es útil para selección de features e interpretabilidad.

In [ ]:
# ── Dropdown interactivo: importar importancias por modelo ───────────────────
models_fi = {
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_gbm,
    "XGBoost": pipe_xgb,
}

feat_names = list(X_train.columns)
traces = []
buttons = []

for i, (nombre, pipe) in enumerate(models_fi.items()):
    imp = pipe.named_steps["model"].feature_importances_
    fi_df = pd.DataFrame({"feature": feat_names, "importance": imp}).sort_values(
        "importance"
    )
    traces.append(
        go.Bar(
            x=fi_df["importance"],
            y=fi_df["feature"],
            orientation="h",
            visible=(i == 0),
            name=nombre,
            marker_color="#4C72B0",
        )
    )
    vis = [j == i for j in range(len(models_fi))]
    buttons.append(
        dict(
            label=nombre,
            method="update",
            args=[{"visible": vis}, {"title": f"Importancia de Features — {nombre}"}],
        )
    )

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Importancia de Features — {list(models_fi.keys())[0]}",
    xaxis_title="Importancia",
    yaxis_title="Feature",
    template="simple_white",
    height=520,
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.18,
            yanchor="top",
        )
    ],
)
fig.show()

> ❓ **Pregunta:** ¿Por qué V14 y V17 podrían ser las features más importantes? ¿Qué podría significar en el contexto de detección de fraude?

#### Pros y Contras — Bagging

| | |
|---|---|
| ✅ Paralelizable: los árboles se entrenan de forma independiente | ❌ No reduce el sesgo (bias) del modelo base |
| ✅ Reduce la varianza y el sobreajuste | ❌ Puede ser lento con datasets muy grandes |
| ✅ El OOB Score evita necesitar un set de validación extra | ❌ Menor interpretabilidad que un árbol individual |

---

## Boosting

> ❓ **Pregunta:** Cuando un estudiante repasa para un examen, ¿debería practicar más las preguntas que ya domina o las que le cuestan? ¿Cómo se relaciona esto con Boosting?

### Concepto

**Boosting** entrena modelos de forma **iterativa y secuencial**: cada modelo se enfoca en los errores del anterior. Los primeros modelos capturan los patrones generales; los siguientes se especializan en los casos difíciles.

<br>
<div align='center'>
    <img src='https://github.com/MDS7202/MDS7202/blob/main/recursos/2023-01/21_Ensamblaje/ensemble_boosting.png?raw=true' width=750/>
</div>
<div align='center'>
    Fuente: <a href='https://en.wikipedia.org/wiki/Boosting_(machine_learning)'>Boosting — Wikipedia</a>
</div>
<br>

El modelo final es una combinación aditiva de todos los modelos entrenados:
$$F_M(x) = \sum_{m=1}^{M} \nu \cdot f_m(x)$$

donde $\nu$ es el **learning rate** que controla cuánto corrige cada modelo. En cada paso, el nuevo *weak learner* $f_m$ se entrena sobre los **residuales** (errores) del modelo anterior, no sobre las etiquetas originales.

Una forma intuitiva de entenderlo: imagina un golfista que corrige su tiro en cada jugada hasta meter la pelota en el hoyo.

![Golf GBM](https://explained.ai/gradient-boosting/images/golf-MSE.png)

### AdaBoost

El primer algoritmo de Boosting ampliamente adoptado. En lugar de ajustar residuales, **aumenta el peso** de los ejemplos mal clasificados en cada iteración, forzando al modelo siguiente a enfocarse en ellos.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

pipe_ada = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        (
            "model",
            AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=44),
        ),
    ]
)
pipe_ada.fit(X_train, y_train)
y_pred_ada = pipe_ada.predict(X_test)
plot_cm(y_test, y_pred_ada, "AdaBoost")

### Gradient Boosting — HistGradientBoosting

`HistGradientBoostingClassifier` es la implementación de sklearn inspirada en LightGBM. Usa histogramas para acelerar el entrenamiento y soporta **early stopping**: detiene el entrenamiento automáticamente cuando la métrica de validación deja de mejorar.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

pipe_gbm = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        (
            "model",
            HistGradientBoostingClassifier(
                random_state=44,
                l2_regularization=10,
                max_depth=50,
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
                scoring="f1",
            ),
        ),
    ]
)
pipe_gbm.fit(X_train, y_train)
y_pred_gbm = pipe_gbm.predict(X_test)
plot_cm(y_test, y_pred_gbm, "HistGradientBoosting")

#### Early Stopping: curva de entrenamiento

La curva siguiente muestra cómo evolucionan los scores de entrenamiento y validación en cada iteración, y cuándo el algoritmo decide parar.

In [ ]:
gbm_model = pipe_gbm.named_steps["model"]
iters = list(range(1, len(gbm_model.train_score_) + 1))

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=iters,
        y=gbm_model.train_score_,
        name="Entrenamiento",
        line=dict(color="#4C72B0"),
    )
)
fig.add_trace(
    go.Scatter(
        x=iters,
        y=gbm_model.validation_score_,
        name="Validación",
        line=dict(color="#DD4444", dash="dash"),
    )
)
fig.add_vline(
    x=gbm_model.n_iter_,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"Stop: iter {gbm_model.n_iter_}",
    annotation_position="top left",
)
fig.update_layout(
    title="Early Stopping — HistGradientBoosting",
    xaxis_title="Iteración",
    yaxis_title="F1-Score",
    template="simple_white",
    legend=dict(x=0.75, y=0.05),
)
fig.show()

> ❓ **Pregunta:** ¿Qué ocurriría si aumentamos demasiado `max_iter` sin early stopping? ¿Y si el `learning_rate` es muy alto?

#### Pros y Contras — Boosting

| | |
|---|---|
| ✅ Alta precisión: reduce bias y varianza | ❌ Entrenamiento secuencial: más lento que Bagging |
| ✅ Maneja valores faltantes (según implementación) | ❌ Más propenso a sobreajuste si no se controla |
| ✅ Genera importancia de features | ❌ Requiere ajuste cuidadoso de hiperparámetros |

---

### XGBoost

> ❓ **Pregunta:** ¿Por qué un algoritmo publicado en 2016 sigue dominando competencias de ML en 2024?

**XGBoost** es una implementación optimizada de Gradient Boosting que revolucionó las competencias de Machine Learning. Sus ventajas clave:

- **Velocidad:** entrenamiento paralelo usando histogramas.
- **Regularización L1 y L2** incorporada.
- **Manejo nativo de valores faltantes.**
- Soporte para GPU.

In [ ]:
from xgboost import XGBClassifier

pipe_xgb = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        (
            "model",
            XGBCla 0.87 ssifier(
                n_estimators=500,
                learning_rate=0.05,
                max_depth=5,
                eval_metric="logloss",
                random_state=44,
                n_jobs=-1,
            ),
        ),
    ]
)
pipe_xgb.fit(X_train, y_train)
y_pred_xgb = pipe_xgb.predict(X_test)
plot_cm(y_test, y_pred_xgb, "XGBoost")

---

## Stacking (Stacked Generalization)

> ❓ **Pregunta:** Si tienes tres expertos con fortalezas distintas, ¿cómo combinarías sus opiniones? ¿Les darías igual peso a todos?

**Stacking** combina modelos **heterogéneos**: un conjunto de modelos base (*level-0*) generan predicciones, y un **meta-learner** (*level-1*) aprende a combinarlas de forma óptima.

```
             ┌─ DecisionTree ──────────┐
             │                         │
Datos ───────┤─ Random Forest ─────────┼──► Meta-learner ──► Predicción final
             │                         │   (LogisticRegression)
             └─ GBM ───────────────────┘
```

Para evitar *data leakage*, las predicciones de los modelos base se generan con **cross-validation**: cada modelo predice sobre el fold en que no fue entrenado. El meta-learner suele ser simple (regresión logística) para evitar sobreajuste.

In [ ]:
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

estimators = [
    ("dt", DecisionTreeClassifier(random_state=44)),
    ("rf", RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=44)),
    ("gbm", HistGradientBoostingClassifier(random_state=44, max_iter=200)),
]

pipe_stack = Pipeline(
    [
        (
            "pre",
            ColumnTransformer(
                [("sc", StandardScaler(), ["Amount"])], remainder="passthrough"
            ),
        ),
        (
            "model",
            StackingClassifier(
                estimators=estimators,
                final_estimator=LogisticRegression(),
                cv=5,
                n_jobs=-1,
            ),
        ),
    ]
)
pipe_stack.fit(X_train, y_train)
y_pred_stack = pipe_stack.predict(X_test)
plot_cm(y_test, y_pred_stack, "Stacking")

### Curva Precision-Recall

Para datasets muy desbalanceados, la curva **Precision-Recall** es más informativa que la ROC. El área bajo la curva (**AP — Average Precision**) resume el rendimiento en todos los umbrales posibles.

> ❓ **Pregunta:** ¿Por qué en fraude bancario nos importa más el Recall que la Precisión? ¿Y cuándo podría importar más la Precisión?

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score

modelos_proba = {
    "Decision Tree": pipe_dt,
    "AdaBoost": pipe_ada,
    "Random Forest": pipe_rf,
    "HistGradientBoosting": pipe_gbm,
    "XGBoost": pipe_xgb,
}
colores = ["#999999", "#FF7F0E", "#2CA02C", "#1F77B4", "#D62728"]

fig = go.Figure()
for (nombre, pipe), color in zip(modelos_proba.items(), colores):
    y_proba = pipe.predict_proba(X_test)[:, 1]
    prec, rec, _ = precision_recall_curve(y_test, y_proba)
    ap = average_precision_score(y_test, y_proba)
    fig.add_trace(
        go.Scatter(
            x=rec,
            y=prec,
            name=f"{nombre}  (AP={ap:.3f})",
            mode="lines",
            line=dict(color=color, width=2),
            hovertemplate="Recall=%{x:.3f}<br>Precisión=%{y:.3f}<extra>"
            + nombre
            + "</extra>",
        )
    )

# Baseline aleatorio
base = labels.mean()
fig.add_hline(
    y=base,
    line_dash="dot",
    line_color="gray",
    annotation_text=f"Baseline aleatorio ({base:.3f})",
    annotation_position="bottom right",
)

fig.update_layout(
    title="Curva Precision-Recall — todos los modelos",
    xaxis_title="Recall",
    yaxis_title="Precisión",
    template="simple_white",
    legend=dict(x=1.01, y=1),
    width=780,
    height=480,
)
fig.show()

---

## Comparación de Modelos

In [ ]:
modelos = {
    "Decision Tree": (pipe_dt, y_pred_dt),
    "AdaBoost": (pipe_ada, y_pred_ada),
    "Random Forest": (pipe_rf, y_pred_rf),
    "HistGradientBoosting": (pipe_gbm, y_pred_gbm),
    "XGBoost": (pipe_xgb, y_pred_xgb),
    "Stacking": (pipe_stack, y_pred_stack),
}

from sklearn.metrics import f1_score, precision_score, recall_score

rows = []
for nombre, (_, y_p) in modelos.items():
    rows.append(
        {
            "Modelo": nombre,
            "F1 Fraude": round(f1_score(y_test, y_p), 4),
            "Precision": round(precision_score(y_test, y_p), 4),
            "Recall": round(recall_score(y_test, y_p), 4),
        }
    )

df_res = pd.DataFrame(rows).sort_values("F1 Fraude")
df_res

In [ ]:
px.bar(
    df_res,
    x="F1 Fraude",
    y="Modelo",
    orientation="h",
    color="F1 Fraude",
    color_continuous_scale="Blues",
    text="F1 Fraude",
    title="Comparación de Modelos — F1-Score clase Fraude",
    template="simple_white",
    height=400,
).update_layout(yaxis={"categoryorder": "total ascending"}, coloraxis_showscale=False)

> ❓ **Pregunta:** ¿Cuál modelo pondrías en producción? ¿Solo el F1-Score debería importar, o hay otros factores relevantes?

---

**Referencias**

- [1] Burkov, A. *The Hundred-Page Machine Learning Book*. Capítulo 7.5.
- [2] Surowiecki, J. *The Wisdom of Crowds* (2004).
- [3] [Gradient Boosting Explained — explained.ai](https://explained.ai/gradient-boosting/descent.html)
- [4] [XGBoost Documentation](https://xgboost.readthedocs.io/)
- [5] [sklearn: Ensemble Methods](https://scikit-learn.org/stable/modules/ensemble.html)